# 实验六：minGPT-2

**课程**：深度学习导论  
**核心交付物**：可运行的 GPT-2 级别 Transformer 解码器 + 权重对齐验证 + 下游微调 + 自回归生成

**AI使用说明**：本实验允许并鼓励负责任地使用 AI 辅助理解、查文档、调试、润色表达。不设抽查答辩、不禁止长段 AI 生成。

## 1. 环境与依赖

In [1]:
import random, math, os, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'使用设备：{device}')

try:
    from transformers import GPT2LMHeadModel, GPT2Tokenizer
except Exception as e:
    print(f'transformers 未安装或导入失败：{e}')
    GPT2LMHeadModel = GPT2Tokenizer = None

try:
    from datasets import load_dataset
except Exception as e:
    print(f'datasets 未安装或导入失败：{e}')
    load_dataset = None


使用设备：cuda


c:\Users\njb18\miniconda3\envs\deeplearning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


datasets 未安装或导入失败：No module named 'datasets'


## 2. 数学基础与推导

本节集中给出 Lab 6 所需的全部数学工具。

### 2.1 GPT-2 架构与 Lab 5 的差分

| 维度 | Lab 5（Post-LN Encoder） | Lab 6（Pre-LN Decoder） |
|------|--------------------------|-------------------------|
| LayerNorm 位置 | 残差相加**之后** | 残差相加**之前** |
| 位置编码 | Sinusoidal（固定） | Learned（可学习） |
| 激活函数 | ReLU | GELU |

**Pre-LN 残差路径**（$l$ 为层索引，$\mathbf{x}$ 为输入）：

$$\mathbf{x}' = \mathbf{x} + \text{MHA}(\text{LN}(\mathbf{x}))$$

$$\mathbf{x}'' = \mathbf{x}' + \text{FFN}(\text{LN}(\mathbf{x}'))$$

**Post-LN 残差路径**（Lab 5 形式，供对比）：

$$\mathbf{x}' = \text{LN}(\mathbf{x} + \text{MHA}(\mathbf{x}))$$

### 2.2 自回归语言建模的概率基础

因果掩码下的条件概率链式分解：

$$P(w_{1:L}) = \prod_{t=1}^{L} P(w_t \mid w_{<t})$$

训练目标为最小化交叉熵损失：

$$\mathcal{L}_{LM} = -\frac{1}{L} \sum_{t=1}^{L} \log P(w_t \mid w_{<t}; \theta)$$

**Teacher Forcing**：训练时以真实 token $w_{<t}$ 作为上下文；推理时以模型自身生成的 token 作为上下文（存在训练-推理分布偏移）。

### 2.3 采样策略的数学定义

设第 $t$ 步的 logits 向量为 $\mathbf{z} \in \mathbb{R}^{|V|}$。

**Greedy Decoding**：$w_t = \arg\max_{w} P(w \mid w_{<t})$

**Temperature Sampling**（$\tau > 0$）：
$$P'(w) = \frac{\exp(z_w / \tau)}{\sum_j \exp(z_j / \tau)}$$
$\tau \to 0$ 退化为 Greedy；$\tau \to \infty$ 趋向均匀分布。

**Top-k Sampling**：保留概率最高的 $k$ 个候选，其余置 $-\infty$，再重新归一化。

**Top-p（Nucleus）Sampling**：按概率降序排列，保留累积概率恰好超过 $p$ 的最小候选集，再重新归一化。

### 2.4 Decoder-only 做判别任务的归纳偏置分析

**【请填写】** 在下方补全三个层面的论证：

$$
\begin{aligned}
1.\; &\text{注意力方向性：} \quad \dots \\
2.\; &\text{预训练目标错配：} \quad \dots \\
3.\; &\text{表征提取位置非对称性：} \quad \dots
\end{aligned}
$$

*（在此填写）*

## 3. GPT-2 架构实现

### 3.1 Learned Positional Embedding

GPT-2 使用可学习的位置编码，而非 Lab 5 中的 Sinusoidal PE。
实现 `nn.Embedding(max_position, d_model)` 的位置编码层，`forward` 中根据序列长度切片并返回位置嵌入。

**【请填写】**：在下方围栏内实现 `LearnedPositionalEmbedding` 类。

In [ ]:
class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_position: int, d_model: int):
        super().__init__()
        # ==========================
        # 【请填写】实现可学习位置编码
        # ==========================
        # self.embedding = ...
        # ==========================

    def forward(self, seq_len: int) -> 'torch.Tensor':
        # ==========================
        # 【请填写】返回前 seq_len 个位置的嵌入向量
        # ==========================
        # return self.embedding(positions)
        # ==========================
        pass


In [ ]:
# 【维度自检】如果此 cell 报错，说明你的实现张量形状错误
_pe = LearnedPositionalEmbedding(max_position=1024, d_model=768)
_out = _pe(10)
assert _out.shape == (10, 768), f'shape error: {_out.shape}'
print('=> LearnedPositionalEmbedding 维度校验通过！')


### 3.2 因果掩码

**【请填写】**：在下方围栏内实现 `make_causal_mask` 函数。

In [ ]:
def make_causal_mask(seq_len: int, device: 'torch.device') -> 'torch.Tensor':
    """返回形状 (1, 1, seq_len, seq_len) 的布尔因果掩码，True 表示被遮蔽。"""
    # ==========================
    # 【请填写】实现因果掩码
    # ==========================
    # mask = ...
    # ==========================
    pass


In [ ]:
# 【维度自检】
_mask = make_causal_mask(4, device=torch.device('cpu'))
assert _mask.shape == (1, 1, 4, 4), f'shape error: {_mask.shape}'
assert _mask[0, 0, 0, 1].item() == True,  '位置 (0,1) 应被遮蔽'
assert _mask[0, 0, 1, 0].item() == False, '位置 (1,0) 不应被遮蔽'
print('=> make_causal_mask 维度与逻辑校验通过！')


### 3.3 带因果掩码的多头自注意力

只需完成 Multi-Head 的 split/merge 部分。

**【请填写】**：在下方围栏内补全 `MultiHeadSelfAttention.forward` 方法。

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.c_attn = nn.Linear(d_model, 3 * d_model)
        self.c_proj = nn.Linear(d_model, d_model)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x: 'torch.Tensor', mask: 'torch.Tensor' = None) -> 'torch.Tensor':
        B, T, C = x.shape
        # ==========================
        # 【请填写】实现多头自注意力前向传播
        # 提示：
        # 1. c_attn 投影后 split 为 Q/K/V，view+transpose 拆分多头
        #    形状变换：(B, T, C) -> (B, n_heads, T, d_k)
        # 2. 参考 Lab 5 §4-5 的 Scaled Dot-Product Attention 实现
        # ==========================
        # Q, K, V = ...
        # attn_weights = ...
        # out = ...
        # ==========================
        pass


In [ ]:
# 【维度自检】
_mha = MultiHeadSelfAttention(d_model=64, n_heads=4)
_x = torch.randn(2, 8, 64)
_mask = make_causal_mask(8, device=torch.device('cpu'))
_out = _mha(_x, _mask)
assert _out.shape == (2, 8, 64), f'shape error: {_out.shape}'
print('=> MultiHeadSelfAttention 维度校验通过！')


### 3.4 GPT-2 Block（Pre-LN）

这是 GPT-2 与 Lab 5 Post-LN TransformerBlock 的核心区别。
只需在 `forward` 中实现正确的 Pre-LN 残差路径。
注意 FFN 使用 GELU 而非 ReLU。

**【请填写】**：在下方围栏内实现 `GPT2Block.forward` 方法。

In [ ]:
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(approximate='tanh'),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: 'torch.Tensor', mask: 'torch.Tensor' = None) -> 'torch.Tensor':
        # ==========================
        # 【请填写】实现 Pre-LN 残差路径
        # ==========================
        # x = ...
        # x = ...
        # ==========================
        pass


In [ ]:
# 【维度自检】
_blk = GPT2Block(d_model=64, n_heads=4, d_ff=256)
_x = torch.randn(2, 8, 64)
_mask = make_causal_mask(8, device=torch.device('cpu'))
_out = _blk(_x, _mask)
assert _out.shape == (2, 8, 64), f'shape error: {_out.shape}'
print('=> GPT2Block 维度校验通过！')


### 3.5 完整 minGPT-2 模型组装

将 Token Embedding、Learned Positional Embedding、多层 GPT-2 Block、最终 LayerNorm 和 LM Head 组装为完整模型。

**【请填写】**：在下方围栏内实现 `minGPT2.forward` 方法。

In [ ]:
class minGPT2(nn.Module):
    def __init__(self, vocab_size: int, max_position: int, d_model: int,
                 n_heads: int, n_layers: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = LearnedPositionalEmbedding(max_position, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            GPT2Block(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids: 'torch.Tensor') -> 'torch.Tensor':
        """input_ids: (B, T) -> logits: (B, T, vocab_size)"""
        B, T = input_ids.shape
        # ==========================
        # 【请填写】实现完整前向传播
        # 提示：
        # 1. x = self.drop(self.tok_emb(input_ids) + self.pos_emb(T))
        # 2. 构造因果掩码，逐层通过 self.blocks
        # 3. logits = self.lm_head(self.ln_f(x))
        # ==========================
        # x = ...
        # mask = ...
        # for block in self.blocks: ...
        # logits = ...
        # ==========================
        pass


In [ ]:
# 【维度自检】
_cfg = dict(vocab_size=50257, max_position=1024, d_model=64,
            n_heads=4, n_layers=2, d_ff=256)
_model = minGPT2(**_cfg)
_ids = torch.randint(0, 50257, (2, 16))
_logits = _model(_ids)
assert _logits.shape == (2, 16, 50257), f'shape error: {_logits.shape}'
print('=> minGPT2 维度校验通过！')


## 4. 权重加载与数值对齐验证

### 4.1 HuggingFace 权重映射

HuggingFace GPT-2 使用 `Conv1D`，权重形状为 `(in, out)`，与 PyTorch `nn.Linear` 的 `(out, in)` 相反，加载时需要转置。

| HuggingFace 键 | 本实现键 | 处理方式 |
|----------------|----------|----------|
| `transformer.wte.weight` | `tok_emb.weight` | 直接复制 |
| `transformer.wpe.weight` | `pos_emb.embedding.weight` | 直接复制 |
| `transformer.h.{i}.attn.c_attn.weight` | `blocks.{i}.attn.c_attn.weight` | **转置** |

**其余映射请自行通过 `hf_model.state_dict().keys()` 探索补全。**
提示：共需映射约 150 个参数张量（12 层 × 每层 12 个参数 + 全局 6 个）。

**【请填写】**：在下方围栏内实现 `load_hf_weights` 函数。

In [ ]:
# 【探索】打印 HuggingFace 与自定义模型的参数键，观察命名规律
if GPT2LMHeadModel is not None:
    hf_model_tmp = GPT2LMHeadModel.from_pretrained('gpt2')
    print("=== HuggingFace GPT-2 参数键（前 20 个）===")
    for i, k in enumerate(hf_model_tmp.state_dict().keys()):
        if i >= 20: break
        print(f"  {k}: {hf_model_tmp.state_dict()[k].shape}")

    print("\n=== 自定义 minGPT2 参数键（前 20 个）===")
    _tmp = minGPT2(vocab_size=50257, max_position=1024, d_model=768,
                   n_heads=12, n_layers=12, d_ff=3072)
    for i, (k, v) in enumerate(_tmp.named_parameters()):
        if i >= 20: break
        print(f"  {k}: {v.shape}")


In [ ]:
def load_hf_weights(model: 'minGPT2', hf_model) -> None:
    """将 HuggingFace GPT2LMHeadModel 的权重加载到 minGPT2 实例中。"""
    hf_sd = hf_model.state_dict()
    n_layers = len(model.blocks)
    # ==========================
    # 【请填写】实现权重映射与加载
    # 提示：
    # 1. 对需要转置的 Conv1D 权重，使用 .t() 后再赋值
    # 2. 用 with torch.no_grad(): param.copy_(value) 安全赋值
    # 3. 按对照表逐一映射，注意 bias 也需要处理
    # ==========================
    pass


### 4.2 前向数值对齐验证

In [ ]:
if GPT2LMHeadModel is not None:
    hf_model = GPT2LMHeadModel.from_pretrained('gpt2')
    hf_model.eval()

    my_model = minGPT2(
        vocab_size=50257, max_position=1024, d_model=768,
        n_heads=12, n_layers=12, d_ff=3072, dropout=0.0
    )
    load_hf_weights(my_model, hf_model)
    my_model.eval()

    test_ids = torch.randint(0, 50257, (1, 16))
    with torch.no_grad():
        my_logits = my_model(test_ids)
        hf_logits = hf_model(test_ids).logits

    max_err = (my_logits - hf_logits).abs().max().item()
    print(f'最大绝对误差：{max_err:.2e}')
    assert max_err < 1e-4, f'数值对齐失败，误差 {max_err:.2e} 超过阈值 1e-4'
    print('=> 数值对齐验证通过！')


**【请填写】** 在下方 code cell 中绘制 logits 对齐误差的直方图，并标注最大绝对误差。

In [ ]:
# ==========================
# 【请填写】绘制 logits 对齐误差直方图
# ==========================
pass


**【结果锚定】** 观察直方图和最大绝对误差数值。

1. 误差分布集中在哪个量级？是否通过了 < 1e-4 的阈值？
2. 如果误差偏大，最可能的原因是什么（提示：检查转置操作和 bias 处理）？

**【请填写】** 在下方 Markdown cell 中作答。

*（在此填写）*

## 5. 下游任务微调

### 5.1 分类头定义

在 GPT-2 最后一个 token 的隐状态上添加线性分类头，支持 Feature Extraction（冻结主干）与全参数微调两种模式。

**【请填写】**：在下方围栏内实现 `GPT2Classifier.forward` 方法。

In [ ]:
class GPT2Classifier(nn.Module):
    def __init__(self, gpt2: 'minGPT2', num_classes: int, freeze_backbone: bool = True):
        super().__init__()
        self.gpt2 = gpt2
        d_model = gpt2.lm_head.in_features
        self.classifier = nn.Linear(d_model, num_classes)
        if freeze_backbone:
            for p in self.gpt2.parameters():
                p.requires_grad = False

    def forward(self, input_ids: 'torch.Tensor') -> 'torch.Tensor':
        # ==========================
        # 【请填写】实现分类前向传播
        # 提示：
        # 1. 获取 GPT-2 最后层隐状态（不经过 lm_head），取最后一个 token 位置的向量
        # ==========================
        # h = ...
        # ==========================
        pass


### 5.2 任务一：SST-2 情感分类

In [ ]:
if load_dataset is not None and GPT2Tokenizer is not None:
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token # GPT-2 默认无 pad_token
    sst2 = load_dataset('glue', 'sst2')
    print(f'SST-2 数据集加载完成；tokenizer 与 sst2 已注入全局命名空间')
else:
    print('跳过 SST-2 数据加载')

In [ ]:
class SST2Dataset(Dataset):
    """SST-2 情感分类数据集。

    Args:
        split: 'train' 或 'validation'
        max_len: tokenizer 最大长度（padding/truncation）
        n: 采样条数上限

    __getitem__ 返回: (input_ids: Tensor[max_len], label: Tensor[scalar])
    """
    def __init__(self, split: str, max_len: int = 64, n: int = 2000):
        # ==========================
        # 【请填写】加载数据并 tokenize
        # 提示：
        # 1. 使用全局变量 tokenizer 和 sst2 数据集
        # 2. tokenizer(..., truncation=True, padding='max_length', max_length=max_len, return_tensors='pt')
        # ==========================
        pass

    def __len__(self):
        # ==========================
        # 【请填写】
        # ==========================
        pass

    def __getitem__(self, i):
        # ==========================
        # 【请填写】返回 (input_ids[i], label[i])
        # ==========================
        pass


In [ ]:
# 【维度自检】SST-2 数据集
_ds = SST2Dataset('train', max_len=64, n=100)
_ids, _label = _ds[0]
assert _ids.shape == (64,), f"input_ids shape error: {_ids.shape}"
assert isinstance(_label.item(), int), "label should be int"
print(f"=> SST2Dataset 自检通过！样本数={len(_ds)}, ids shape={_ids.shape}")


In [ ]:
if load_dataset is not None and GPT2Tokenizer is not None:
    sst2_train = DataLoader(SST2Dataset('train'), batch_size=32, shuffle=True)
    sst2_val = DataLoader(SST2Dataset('validation'), batch_size=64)
    print(f'SST-2 训练集：{len(sst2_train.dataset)}，验证集：{len(sst2_val.dataset)}')

**【请填写】**：在下方围栏内实现 `train_classifier` 训练循环。

In [ ]:
def train_classifier(model, train_loader, val_loader, epochs=3, lr=2e-4):
    """训练分类器，返回 (train_losses: List[float], val_accs: List[float])。

    要求：
    - 使用 AdamW 优化器（仅优化 requires_grad=True 的参数）
    - 使用 CrossEntropyLoss
    - 每个 epoch 结束后在 val_loader 上评估准确率
    - 打印每个 epoch 的 loss 和 val_acc
    - 遇到 NaN loss 时 break 并打印警告
    """
    # ==========================
    # 【请填写】实现完整的训练与验证循环
    # 提示：
    # 1. optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    # 2. 训练循环：zero_grad -> forward -> loss -> backward -> step
    # 3. 验证循环：torch.no_grad() 下计算准确率
    # 4. 返回 (train_losses, val_accs) 两个列表
    # ==========================
    pass


In [ ]:
if load_dataset is not None and GPT2LMHeadModel is not None:
    sst2_clf = GPT2Classifier(my_model, num_classes=2, freeze_backbone=True)
    losses_sst2, accs_sst2 = train_classifier(sst2_clf, sst2_train, sst2_val)
else:
    print('跳过 SST-2 训练')

**【请填写】** 在下方 code cell 中绘制 SST-2 的训练 loss 曲线与验证准确率曲线（双子图）。

In [ ]:
# ==========================
# 【请填写】绘制 SST-2 训练曲线
# ==========================
pass


**【结果锚定】** 记录 SST-2 Feature Extraction 模式下的最终验证准确率。

**【请填写】** 在下方 Markdown cell 中作答。

*（在此填写）*

### 5.3 任务二：AG News 主题分类（四分类）

In [ ]:
if load_dataset is not None and GPT2Tokenizer is not None:
    agnews = load_dataset('ag_news')
    print(f'AG News 数据集加载完成；agnews 已注入全局命名空间')
else:
    print('跳过 AG News 数据加载')

In [ ]:
class AGNewsDataset(Dataset):
    """AG News 主题分类数据集。

    Args:
        split: 'train' 或 'test'
        max_len: tokenizer 最大长度
        n: 采样条数上限

    __getitem__ 返回: (input_ids: Tensor[max_len], label: Tensor[scalar])
    """
    def __init__(self, split: str, max_len: int = 64, n: int = 5000):
        # ==========================
        # 【请填写】加载数据并 tokenize
        # 提示：
        # 1. 使用全局变量 tokenizer 和 agnews 数据集
        # 2. tokenizer(..., truncation=True, padding='max_length', max_length=max_len, return_tensors='pt')
        # ==========================
        pass

    def __len__(self):
        # ==========================
        # 【请填写】
        # ==========================
        pass

    def __getitem__(self, i):
        # ==========================
        # 【请填写】返回 (input_ids[i], label[i])
        # ==========================
        pass


In [ ]:
# 【维度自检】AG News 数据集
_ds = AGNewsDataset('train', max_len=64, n=100)
_ids, _label = _ds[0]
assert _ids.shape == (64,), f"input_ids shape error: {_ids.shape}"
assert _label.item() in range(4), f"label out of range: {_label.item()}"
print(f"=> AGNewsDataset 自检通过！样本数={len(_ds)}, ids shape={_ids.shape}")


In [ ]:
if load_dataset is not None and GPT2Tokenizer is not None:
    ag_train = DataLoader(AGNewsDataset('train'), batch_size=32, shuffle=True)
    ag_val = DataLoader(AGNewsDataset('test', n=1000), batch_size=64)
    print(f'AG News 训练集：{len(ag_train.dataset)}，验证集：{len(ag_val.dataset)}')

In [ ]:
if load_dataset is not None and GPT2LMHeadModel is not None:
    for p in my_model.parameters():
        p.requires_grad = True

    ag_frozen = GPT2Classifier(my_model, num_classes=4, freeze_backbone=True)
    ag_ft     = GPT2Classifier(my_model, num_classes=4, freeze_backbone=False)

    print('=== Feature Extraction ===')
    _, accs_ag_frozen = train_classifier(ag_frozen, ag_train, ag_val)
    print('=== Full Fine-tuning ===')
    _, accs_ag_ft = train_classifier(ag_ft, ag_train, ag_val, lr=5e-5)
    print(f'冻结主干最终 val_acc：{accs_ag_frozen[-1]:.4f}')
    print(f'全参数微调最终 val_acc：{accs_ag_ft[-1]:.4f}')
else:
    print('跳过 AG News 训练')


**【结果锚定】** 对比冻结主干与全参数微调在 AG News 上的准确率差异。

**注意**：本节的实验设置（10 epoch、5e-5 学习率、5000 样本子集）目的是减少运行时间的同时放大 Decoder-only 模型在判别任务上的结构劣势。在工业界的真实配方下全参数微调通常能超过冻结主干 5-15 个百分点，但仍弱于同等条件下的 BERT。

**【请填写】** 在下方 Markdown cell 中作答。

*（在此填写）*

### 5.4 任务三：Shakespeare 文本生成微调

In [ ]:
SHAKESPEARE_URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
SHAKESPEARE_PATH = './data/tinyshakespeare.txt'
os.makedirs('./data', exist_ok=True)
if not os.path.exists(SHAKESPEARE_PATH):
    urllib.request.urlretrieve(SHAKESPEARE_URL, SHAKESPEARE_PATH)
    print('TinyShakespeare 下载完成')

with open(SHAKESPEARE_PATH, encoding='utf-8') as f:
    shakespeare_text = f.read()

USE_SMALL_SUBSET = True  # 设为 False 可使用完整数据集（CPU 需 30-60 分钟）
if USE_SMALL_SUBSET:
    shakespeare_text = shakespeare_text[:50000]
    print('使用前 50000 字符的子集（CPU 友好模式）')
print(f'文本长度：{len(shakespeare_text)} 字符')


In [ ]:
if GPT2Tokenizer is not None:
    shk_ids = tokenizer(shakespeare_text, return_tensors='pt')['input_ids'][0]
    block_size = 128

    class ShakespeareDataset(Dataset):
        def __init__(self, ids, block_size):
            self.ids, self.block_size = ids, block_size

        def __len__(self): return len(self.ids) - self.block_size

        def __getitem__(self, i):
            # ==========================
            # 【请填写】返回 (input_ids, targets)
            # 提示：
            # 1. input  = self.ids[i : i + self.block_size]
            # 2. target = self.ids[i+1 : i + self.block_size + 1]（向右偏移 1）
            # ==========================
            pass

    shk_loader = DataLoader(ShakespeareDataset(shk_ids, block_size), batch_size=16, shuffle=True)
    print(f'Shakespeare 数据集大小：{len(shk_loader.dataset)} 样本')
else:
    print('跳过 Shakespeare 数据集构建')


In [ ]:
# 【维度自检】Shakespeare 数据集
if GPT2Tokenizer is not None:
    _ds = ShakespeareDataset(shk_ids, block_size)
    _x, _y = _ds[0]
    assert _x.shape == (block_size,), f"input shape error: {_x.shape}"
    assert _y.shape == (block_size,), f"target shape error: {_y.shape}"
    assert (_x[1:] == _y[:-1]).all(), "target should be input shifted by 1"
    print(f"=> ShakespeareDataset 自检通过！样本数={len(_ds)}")


**【请填写】**：在下方围栏内实现 `train_lm` 语言模型训练循环。

In [ ]:
def train_lm(model, loader, epochs=2, lr=3e-4):
    """语言模型微调，返回 epoch_losses: List[float]。

    要求：
    - logits 形状 (B, T, V)，targets 形状 (B, T)
    - loss = CrossEntropyLoss(logits.view(-1, V), targets.view(-1))
    - 遇到 NaN loss 时 break 并打印警告
    """
    # ==========================
    # 【请填写】实现完整的语言模型训练循环
    # ==========================
    pass


In [ ]:
if GPT2Tokenizer is not None and GPT2LMHeadModel is not None:
    shk_losses = train_lm(my_model, shk_loader)
else:
    print('跳过 Shakespeare 微调')

**【请填写】** 在下方 code cell 中绘制 Shakespeare 微调的 loss 曲线。

In [ ]:
# ==========================
# 【请填写】绘制 Shakespeare 微调 loss 曲线
# ==========================
pass


## 6. 自回归采样策略实现与消融

### 6.1 四种采样策略

实现 Greedy、Temperature、Top-k、Top-p 四种策略，统一封装在 `generate` 函数中。
生成循环必须设置 `max_new_tokens` 硬性上限，防止无限循环。

**【请填写】**：在下方围栏内实现 `generate` 函数。

In [ ]:
@torch.no_grad()
def generate(model: 'minGPT2', input_ids: 'torch.Tensor', max_new_tokens: int = 100,
             temperature: float = 1.0, top_k: int = 0, top_p: float = 1.0) -> 'torch.Tensor':
    """自回归生成。temperature=1e-6 近似 Greedy；top_k=0, top_p=1.0 为纯 Temperature Sampling。"""
    model.eval()
    ids = input_ids.clone()
    max_pos = model.pos_emb.embedding.weight.shape[0]

    for _ in range(max_new_tokens):
        # ==========================
        # 【请填写】实现自回归生成循环
        # 提示：
        # 1. 每步取 logits 的最后一个位置，除以 temperature
        # 2. top_k > 0 时截断低概率候选
        # 3. top_p < 1.0 时按累积概率截断
        # 4. 用 torch.multinomial 采样下一个 token
        # 5. 注意处理序列长度超出位置编码范围的情况
        # ==========================
        # ids = torch.cat([ids, next_id], dim=1)
        # ==========================
        pass

    return ids


### 6.2 四种策略生成结果对比

In [ ]:
if GPT2Tokenizer is not None and GPT2LMHeadModel is not None:
    prompt = 'To be, or not to be,'
    prompt_ids = tokenizer(prompt, return_tensors='pt')['input_ids'].to(device)

    strategies = [
        ('Greedy',        dict(temperature=1e-6, top_k=0,  top_p=1.0)),
        ('Temperature',   dict(temperature=0.8,  top_k=0,  top_p=1.0)),
        ('Top-k (k=50)',  dict(temperature=1.0,  top_k=50, top_p=1.0)),
        ('Top-p (p=0.9)', dict(temperature=1.0,  top_k=0,  top_p=0.9)),
    ]
    for name, kwargs in strategies:
        out = generate(my_model, prompt_ids, max_new_tokens=80, **kwargs)
        text = tokenizer.decode(out[0], skip_special_tokens=True)
        print(f'\n[{name}]\n{text}')
else:
    print('跳过生成演示')


**【理论锚定】** 对比四种策略的生成文本，从以下角度分析：

1. Greedy Decoding 为何是确定性的？多次运行结果是否相同？
2. Temperature 如何控制 softmax 的"尖锐度"？$\tau < 1$ 与 $\tau > 1$ 分别有何效果？
3. Top-k 与 Top-p 在"保持连贯性"与"增加多样性"之间的权衡差异是什么？

**【请填写】** 在下方 Markdown cell 中作答。

*（在此填写）*

## 7. 架构归纳偏置对比分析

### 7.1 性能汇总表

将本 Lab 的实验结果与 Lab 5 中 BERT 的已知表现进行横向对比（无需重跑 BERT，引用 Lab 5 结论即可）。

| 模型 | SST-2 准确率 | AG News 准确率 | 备注 |
|------|-------------|---------------|------|
| GPT-2（冻结主干） | *（填写）* | *（填写）* | Feature Extraction |
| GPT-2（全参数微调） | — | *（填写）* | Full Fine-tuning |
| BERT（Lab 5 结论） | *（引用）* | *（引用）* | 参考值 |

**【请填写】** 在下方 Markdown cell 中填写上表数据并完成分析。

*（在此填写）*

### 7.2 理论论证

**【理论锚定】** 从注意力方向性、预训练目标错配、表征提取位置非对称性三个层面，
论证 Decoder-only 模型（GPT-2）在分类任务上弱于 Encoder-only 模型（BERT）的根本原因。
每个层面请写 3-5 句完整段落。

**【请填写】** 在下方 Markdown cell 中作答。

*（在此填写）*

## 8. 书面作答

以下问题要求每条 3-5 句完整段落。

### 8.1 HuggingFace Conv1D 与 PyTorch nn.Linear 的转置关系

**【理论锚定】** 解释 HuggingFace GPT-2 实现中 `Conv1D` 权重形状为 `(in, out)` 的历史原因，
以及加载时为何必须转置。

**【请填写】** 在下方 Markdown cell 中作答。

*（在此填写）*

### 8.2 Pre-LN 与 Post-LN 的梯度流差异

**【理论锚定】** 解释 Pre-LN 在深层网络训练中相比 Post-LN 的梯度流优势，
以及为何 GPT-2 选择 Pre-LN。

**【请填写】** 在下方 Markdown cell 中作答。

*（在此填写）*

## 9. 附加题（选做）

以下题目不计入基础分，完成可获得额外加分。

在 `generate` 函数中实现重复惩罚机制：对已生成的 token，将其对应 logit 除以惩罚系数 $\alpha > 1$，抑制生成文本中的逐字重复现象。


*实验六到此结束。请确认所有必填项已完成并提交本 Notebook。*